# Email Data Processing

This notebook reads email files from the `data` directory and organizes them into structured data using Pandas DataFrames

In [1]:
import os
import numpy as np
import pandas as pd
import email
from bs4 import BeautifulSoup

In [4]:
contents = []
labels = []
fnames = []

for root,dirs,files in os.walk("../data/"):
    for file in files:
        fnames.append(file)
        abs_path = os.path.join(root, file)
        try:
            if file.endswith("ipynb"):
                pass
            elif "ham" in abs_path:
                contents.append(abs_path)
                labels.append(0)
            elif "spam" in abs_path:
                contents.append(abs_path)
                labels.append(1)
            else:
                print(f"something happened you didn't expect for file {abs_path}")
        except Exception as e:
            print(f"Error processing {abs_path}: {str(e)}")
            

In [80]:
def get_multipart(msg):
    payloads = msg.get_payload()
    content_out = ""
    if type(payloads) == list:
        for part in payloads:
            content_type = part.get_content_type()
            content_disposition = part.get_content_disposition()
            charset = part.get_content_charset() or 'utf-8'
            
            if content_type == 'text/plain' and content_disposition is None:
                content1 = part.get_payload(decode=True).decode(charset, errors='replace')
                content_out=content_out+content1
            elif content_type == 'text/html' and content_disposition is None:
                soup = BeautifulSoup(part.get_payload(decode=True), 'html.parser')
                content2 = soup.get_text(separator="\n", strip=True)
                content_out=content_out+content2
    else:
        soup = BeautifulSoup(msg.get_payload(decode=True), 'html.parser')
        content_out = soup.get_text(separator="\n", strip=True)
        
    return content_out

In [81]:
def get_txt_from_type(type_, msg):
    if type_ == "text/plain":
        txt = msg.get_payload()
    elif type_ == "text/html":
        soup = BeautifulSoup(msg.get_payload(decode=True), 'html.parser')
        txt = soup.get_text(separator="\n", strip=True)
    else:
        txt = get_multipart(msg)
    return txt

In [ ]:
content_out = ""
for part in payloads:
    content_type = part.get_content_type()
    content_disposition = part.get_content_disposition()
    charset = part.get_content_charset() or 'utf-8'
    
    if content_type == 'text/plain' and content_disposition is None:
        content1 = part.get_payload(decode=True).decode(charset, errors='replace')
        content_out=content_out+content1
    elif content_type == 'text/html' and content_disposition is None:
        soup = BeautifulSoup(part.get_payload(decode=True), 'html.parser')
        content2 = soup.get_text(separator="\n", strip=True)
        content_out=content_out+content2

In [113]:
%%time
i=0
ftext = []
for item in contents:
    with open(item, "r", encoding="latin-1") as f:
        msg = email.message_from_file(f)
        type_ = msg.get_content_type()
        txt = get_txt_from_type(type_, msg)
        ftext.append(txt)

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


CPU times: user 13.7 s, sys: 2.88 s, total: 16.5 s
Wall time: 57.4 s


In [96]:
df = pd.DataFrame({
    'filename':fnames,
    'contents':ftext,
    'target':labels})

In [112]:
df.to_csv("emails.csv",index=True)